## Launching colab kernel

```
colab launch  //third_party/py/torchtitan:torchtitan_colab \
 --xm_resource_alloc=cloud-dynamic/cmcs-xm \
 --accelerator=vl:4x2  \
 --label="$USER - vl4x2 `date '+%Y-%m-%d %H:%M:%S'`"
```

In [1]:
%%writefile /tmp/debug_model.toml
[job]
dump_folder = "./outputs"
description = "Qwen 3 debug model training"

[profiling]
enable_profiling = false
save_traces_folder = "profile_trace"
profile_freq = 100

[metrics]
log_freq = 1
enable_tensorboard = false
save_tb_folder = "tb"

[model]
name = "qwen3"
flavor = "debugmodel"
hf_assets_path = "./assets/hf/Qwen3-0.6B"
# converters = ["float8"]

[optimizer]
name = "AdamW"
lr = 3e-4
eps = 1e-8

[lr_scheduler]
warmup_steps = 2  # lr scheduler warm up, 20% total steps

[training]
local_batch_size = 4
seq_len = 128
max_norm = 1.0  # grad norm clipping
steps = 10
dataset = "c4_test"  # supported datasets: c4_test (2K), c4 (177M)

[parallelism]
data_parallel_replicate_degree = 1
data_parallel_shard_degree = -1
fsdp_reshard_after_forward = "default" # default / never / always
tensor_parallel_degree = 1
context_parallel_degree = 1

[checkpoint]
enable = false
folder = "checkpoint"
interval = 500
last_save_model_only = false
export_dtype = "float16"
async_mode = "disabled" # ["disabled", "async", "async_with_pinned_mem"]

[activation_checkpoint]
mode = "selective"  # ["none", "selective", "full"]
selective_ac_option = "op"  # "int" = ac every positive int layer or 'op', ac based on ops policy

[compile]
enable=false
components = ["model", "loss"]

[quantize.linear.float8]
enable_fsdp_float8_all_gather = false
precompute_float8_dynamic_scale_for_fsdp = false
filter_fqns = ["output"]


Writing /tmp/debug_model.toml


In [2]:

import torchtitan.experiments.tpu.train_minimal

import torchtitan.config
from torchtitan.experiments.tpu import accelerator_device_type as device_type

num_devices = 8
accelerator_device_type=device_type.AcceleratorDeviceType.TPU

config_manager = torchtitan.config.ConfigManager()
config = config_manager.parse_args([
    "--job.config_file=/tmp/debug_model.toml",
    "--model.hf_assets_path=/cns/is-d/home/torch-tpu-xm/torchtitan/tests/assets/tokenizer",
    "--training.dataset_path=/cns/is-d/home/torch-tpu-xm/torchtitan/tests/assets/c4_test",
    "--training.seq_len=128",
    "--training.dataset=c4_test",
    "--parallelism.data_parallel_shard_degree=1",
    f"--parallelism.tensor_parallel_degree={num_devices}"
])
config.training.steps = 20

trainer = torchtitan.experiments.tpu.train_minimal.TrainerMinimal(
    config)


Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
from torchtitan.experiments.tpu import profiling

profiling.run_distributed_with_profiling(
    num_devices=num_devices,
    accelerator_device_type=accelerator_device_type,
    func=trainer.train,
    # combine_profiles=False,
    )

Successfully downloaded jialeic-10907640810715665036 for rank 0
Successfully downloaded jialeic-17274737609583683036 for rank 1
Successfully downloaded jialeic-12544568877311832750 for rank 2
Successfully downloaded jialeic-12014950894573778746 for rank 3
Successfully downloaded jialeic-2786013682762543113 for rank 4
Successfully downloaded jialeic-9149091623859259055 for rank 5
Successfully downloaded jialeic-4741069511727402611 for rank 6
Successfully downloaded jialeic-719067361444551643 for rank 7
Xprof URL: http://xprof/?session_id=jialeic-7457327058696636056
